# 🚀 EasyLLM: Qwen3-4B-Instruct-2507 QLoRA Training Pipeline
**Job ID:** `job-b74a80d9`  
**Target Model:** `Qwen/Qwen3-4B-Instruct-2507`  
**Architecture:** QLoRA 4-bit NormalFloat (NF4) with Rank $r=8$, Alpha $\alpha=16$  
**Optimized Settings:** Sequence Length=512, Batch Size=1, Gradient Accumulation=4, Gradient Checkpointing=Enabled.

This notebook was autonomously configured by **EasyLLM** to train your customized AI on Google Colab (Free T4 / A100 GPU).

In [ ]:
# 1. Install Real Deep Learning & QLoRA Dependencies
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate
!nvidia-smi

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Ingest Dataset (Pre-injected from your EasyLLM job)
RAW_RECORDS = []

formatted_texts = []
for r in RAW_RECORDS:
    msgs = r.get('messages', [])
    turn_text = []
    for m in msgs:
        role = m.get('role', 'user')
        content = m.get('content', '')
        turn_text.append(f'<|im_start|>{role}\n{content}<|im_end|>')
    formatted_texts.append('\n'.join(turn_text))

dataset = Dataset.from_dict({'text': formatted_texts})
print(f'Successfully loaded {len(dataset)} training sequences for Qwen3.')


In [ ]:
# 3. Load Qwen3-4B Base Model in 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()


In [ ]:
# 4. Apply Parameter-Efficient LoRA Adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


In [ ]:
# 5. Execute Supervised Fine-Tuning (SFT)
training_args = TrainingArguments(
    output_dir='./easyllm_qwen3_output',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=0.0002,
    fp16=True,
    logging_steps=1,
    save_strategy='no',
    report_to='none'
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print('Starting Qwen3 QLoRA fine-tuning...')
trainer.train()


In [ ]:
# 6. Export Trained LoRA Adapter Package for EasyLLM
ADAPTER_DIR = './easyllm_adapter'
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

!zip -r easyllm_qwen3_adapter.zip ./easyllm_adapter
from google.colab import files
files.download('easyllm_qwen3_adapter.zip')
print('🎉 Training complete! Upload easyllm_qwen3_adapter.zip back into EasyLLM.')
